**CARREGANDO TABELA**

In [0]:
%sql
SELECT * FROM workspace.default.hospital_data_analysis LIMIT 10;

Patient_ID,Age,Gender,Condition,Procedure,Cost,Length_of_Stay,Readmission,Outcome,Satisfaction
1,45,Female,Heart Disease,Angioplasty,15000,5,No,Recovered,4
2,60,Male,Diabetes,Insulin Therapy,2000,3,Yes,Stable,3
3,32,Female,Fractured Arm,X-Ray and Splint,500,1,No,Recovered,5
4,75,Male,Stroke,CT Scan and Medication,10000,7,Yes,Stable,2
5,50,Female,Cancer,Surgery and Chemotherapy,25000,10,No,Recovered,4
6,68,Male,Hypertension,Medication and Counseling,1000,2,No,Stable,4
7,55,Female,Appendicitis,Appendectomy,8000,4,No,Recovered,3
8,40,Male,Fractured Leg,Cast and Physical Therapy,3000,6,No,Recovered,4
9,70,Female,Heart Attack,Cardiac Catheterization,18000,8,Yes,Stable,2
10,25,Male,Allergic Reaction,Epinephrine Injection,100,1,No,Recovered,5


**GERANDO SUMMARY**

In [0]:
%python
df = spark.table("workspace.default.hospital_data_analysis")
display(df.summary())

summary,Patient_ID,Age,Gender,Condition,Procedure,Cost,Length_of_Stay,Readmission,Outcome,Satisfaction
count,984,984,984,984,984,984,984,984,984,984
mean,500.3292682926829,53.75406504065041,null,null,null,8367.479674796748,37.66361788617886,null,null,3.5985772357723578
stddev,288.9795307515055,14.941135218153875,null,null,null,7761.990976300389,19.5958052251366,null,null,0.8830021435326754
min,1,25,Female,Allergic Reaction,Angioplasty,100,1,No,Recovered,2
25%,250,45,null,null,null,1000,21,null,null,3
50%,500,55,null,null,null,6000,38,null,null,4
75%,750,65,null,null,null,15000,54,null,null,4
max,1000,78,Male,Stroke,X-Ray and Splint,25000,76,Yes,Stable,5


**CRIANDO TABELAS PARA GERAR O ESQUEMA ESTRELA**

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

# 1. Carregando a tabela original (Raw/Bronze)
df_raw = spark.table("workspace.default.hospital_data_analysis")

# 2. Criando a Dimensão Paciente (Dim_Paciente)
# Selecionamos apenas os dados únicos do paciente
df_dim_paciente = df_raw.select("Patient_ID", "Age", "Gender").dropDuplicates(["Patient_ID"])

# 3. Criando a Dimensão Clínica/Procedimento (Dim_Procedimento)
# Como Condition e Procedure não têm um ID próprio, vamos criar um identificador único
df_dim_procedimento = df_raw.select("Condition", "Procedure").dropDuplicates()
df_dim_procedimento = df_dim_procedimento.withColumn("Procedimento_ID", monotonically_increasing_id())

# 4. Criando a Tabela Fato Internação (Fato_Internacao)
# Fazemos um "join" para trazer o Procedimento_ID para a tabela Fato
df_fato_internacao = df_raw.join(df_dim_procedimento, on=["Condition", "Procedure"], how="left")

# Selecionamos as chaves e as métricas para a Fato
df_fato_internacao = df_fato_internacao.select(
    "Patient_ID", 
    "Procedimento_ID", 
    "Cost", 
    "Length_of_Stay", 
    "Readmission", 
    "Outcome", 
    "Satisfaction"
)

# 5. Salvando as tabelas prontas no Databricks (Camada Silver/Gold)
df_dim_paciente.write.mode("overwrite").saveAsTable("workspace.default.dim_paciente")
df_dim_procedimento.write.mode("overwrite").saveAsTable("workspace.default.dim_procedimento")
df_fato_internacao.write.mode("overwrite").saveAsTable("workspace.default.fato_internacao")

display(df_fato_internacao)

Patient_ID,Procedimento_ID,Cost,Length_of_Stay,Readmission,Outcome,Satisfaction
1,0,15000,5,No,Recovered,4
2,1,2000,3,Yes,Stable,3
3,4,500,1,No,Recovered,5
4,13,10000,7,Yes,Stable,2
5,10,25000,10,No,Recovered,4
6,11,1000,2,No,Stable,4
7,8,8000,4,No,Recovered,3
8,3,3000,6,No,Recovered,4
9,7,18000,8,Yes,Stable,2
10,12,100,1,No,Recovered,5


### RESPONDEDO AS PERGUNTAS DE NEGÓCIO

1. Qual condição de saúde gera o maior tempo médio de internação?

In [0]:
%sql
-- Pergunta 1: Tempo médio de internação por condição clínica
SELECT 
    d.Condition AS Condicao_Clinica, 
    ROUND(AVG(f.Length_of_Stay), 2) AS Tempo_Medio_Internacao_Dias
FROM 
    workspace.default.fato_internacao f
JOIN 
    workspace.default.dim_procedimento d ON f.Procedimento_ID = d.Procedimento_ID
GROUP BY 
    d.Condition
ORDER BY 
    Tempo_Medio_Internacao_Dias DESC;

Condicao_Clinica,Tempo_Medio_Internacao_Dias
Cancer,42.65
Prostate Cancer,41.58
Heart Attack,41.0
Stroke,40.33
Fractured Leg,39.0
Heart Disease,38.22
Osteoarthritis,37.86
Appendicitis,37.44
Kidney Stones,36.46
Diabetes,36.09


Databricks visualization. Run in Databricks to view.

**Discussão:** Os dados revelam que as condições oncológicas (Câncer e Câncer de Próstata) e eventos cardiovasculares ou neurológicos agudos (Ataque Cardíaco e AVC) são os maiores consumidores de tempo de leito hospitalar, com médias superiores a 40 dias. Para a gestão estratégica do hospital, isso indica que o planejamento de capacidade, a alocação de equipes de enfermagem especializadas e a previsão de custos devem focar prioritariamente nestes setores de alta complexidade. Em contrapartida, condições como reações alérgicas e fraturas de braço apresentam giros de leito mais rápidos, permitindo maior rotatividade e liberação de espaço na unidade.

2. Qual é o custo médio de cada tipo de procedimento realizado?

In [0]:
%sql
-- Pergunta 2: Custo médio por tipo de procedimento
SELECT 
    d.Procedure AS Tipo_Procedimento, 
    ROUND(AVG(f.Cost), 2) AS Custo_Medio
FROM 
    workspace.default.fato_internacao f
JOIN 
    workspace.default.dim_procedimento d ON f.Procedimento_ID = d.Procedimento_ID
GROUP BY 
    d.Procedure
ORDER BY 
    Custo_Medio DESC;

Tipo_Procedimento,Custo_Medio
Surgery and Chemotherapy,25000.0
Radiation Therapy,20000.0
Cardiac Catheterization,18000.0
Angioplasty,15000.0
Delivery and Postnatal Care,12000.0
CT Scan and Medication,10000.0
Appendectomy,8000.0
Lithotripsy,6000.0
Physical Therapy and Pain Management,4000.0
Cast and Physical Therapy,3000.0


Databricks visualization. Run in Databricks to view.

**Discussão:** O gráfico demonstra que os procedimentos associados a áreas de alta complexidade, como "Surgery and Chemotherapy" (Cirurgia e Quimioterapia) e "Radiation Therapy" (Radioterapia), representam o maior encargo financeiro, atingindo patamares em torno dos 25.000 e 20.000, respetivamente. Intervenções cardiológicas, como o Cateterismo Cardíaco e a Angioplastia, também se destacam com custos elevados (entre 15.000 e 18.000). Em contrapartida, tratamentos de rotina ou intervenções simples (como injeções de epinefrina, medicação e raio-X) têm um custo médio residual. Do ponto de vista da gestão e administração hospitalar, esta análise é crucial, pois indica que a alocação de orçamento e o planeamento financeiro devem focar-se substancialmente nas alas de oncologia e cardiologia, que absorvem a maior fatia do capital.

3. Existe uma relação entre o tempo de estadia e a satisfação?

In [0]:
%sql
-- Pergunta 3: Relação (correlação simples) entre tempo de estadia e satisfação
-- Agrupando por nível de satisfação para ver se o tempo médio de estadia muda
SELECT 
    f.Satisfaction AS Nivel_Satisfacao,
    COUNT(*) AS Quantidade_Pacientes,
    ROUND(AVG(f.Length_of_Stay), 2) AS Tempo_Medio_Internacao_Dias
FROM 
    workspace.default.fato_internacao f
GROUP BY 
    f.Satisfaction
ORDER BY 
    f.Satisfaction DESC;

Nivel_Satisfacao,Quantidade_Pacientes,Tempo_Medio_Internacao_Dias
5,132,34.25
4,458,37.5
3,261,38.15
2,133,40.67


Databricks visualization. Run in Databricks to view.

**Discussão:** Sim, os dados demonstram uma relação inversamente proporcional entre o tempo de estadia e a satisfação do paciente. Pacientes que reportam o nível máximo de satisfação (nível 5) apresentam o menor tempo médio de internamento (34,25 dias). À medida que o tempo de estadia aumenta, a satisfação cai progressivamente, atingindo o nível mais baixo (nível 2) nos pacientes com os internamentos mais longos (média de 40,67 dias). Para a gestão hospitalar, isto indica que otimizar os processos de recuperação e acelerar as altas seguras não é apenas uma questão de redução de custos, mas também um fator crítico para melhorar a experiência e a perceção de qualidade por parte dos doentes.

4. Qual faixa etária tem maior taxa de readmissão?

In [0]:
%sql
-- Pergunta 4: Taxa de readmissão por faixa etária
-- Primeiro, criamos as faixas etárias usando CASE, depois contamos as readmissões
WITH FaixasEtarias AS (
    SELECT 
        CASE 
            WHEN p.Age < 30 THEN 'Menos de 30'
            WHEN p.Age BETWEEN 30 AND 50 THEN 'De 30 a 50'
            WHEN p.Age BETWEEN 51 AND 65 THEN 'De 51 a 65'
            ELSE 'Mais de 65' 
        END AS Faixa_Etaria,
        f.Readmission
    FROM 
        workspace.default.fato_internacao f
    JOIN 
        workspace.default.dim_paciente p ON f.Patient_ID = p.Patient_ID
)
SELECT 
    Faixa_Etaria,
    COUNT(CASE WHEN Readmission = 'Yes' THEN 1 END) AS Qtd_Readmissoes,
    COUNT(*) AS Total_Pacientes,
    ROUND((COUNT(CASE WHEN Readmission = 'Yes' THEN 1 END) * 100.0) / COUNT(*), 2) AS Taxa_Readmissao_Porcentagem
FROM 
    FaixasEtarias
GROUP BY 
    Faixa_Etaria
ORDER BY 
    Taxa_Readmissao_Porcentagem DESC;

Faixa_Etaria,Qtd_Readmissoes,Total_Pacientes,Taxa_Readmissao_Porcentagem
Mais de 65,100,231,43.29
De 51 a 65,100,391,25.58
De 30 a 50,64,296,21.62
Menos de 30,0,66,0.00


**Discussão:** Os dados indicam que a faixa etária "Mais de 65" anos apresenta, de forma destacada, a maior taxa de readmissão hospitalar (43,29%), representando quase o dobro da faixa etária seguinte ("De 51 a 65" anos, com 25,58%). Em contrapartida, os pacientes mais jovens ("Menos de 30" anos) não registaram qualquer readmissão na amostra (0,00%). Do ponto de vista da gestão e qualidade do serviço hospitalar, isto sinaliza que os pacientes idosos são substancialmente mais vulneráveis a complicações pós-alta. Para mitigar este problema (que gera custos acrescidos e diminui a disponibilidade de camas), o hospital deve investir em protocolos de alta médica mais rigorosos para esta faixa etária, bem como em programas de acompanhamento domiciliário ou contacto proativo após a saída da unidade.

5. Como o gênero impacta o desfecho do tratamento?

In [0]:
%sql
-- Pergunta 5: Impacto do gênero no desfecho (Outcome)
SELECT 
    p.Gender AS Genero,
    f.Outcome AS Desfecho,
    COUNT(*) AS Quantidade
FROM 
    workspace.default.fato_internacao f
JOIN 
    workspace.default.dim_paciente p ON f.Patient_ID = p.Patient_ID
GROUP BY 
    p.Gender, f.Outcome
ORDER BY 
    p.Gender, Quantidade DESC;

Genero,Desfecho,Quantidade
Female,Recovered,328
Female,Stable,196
Male,Recovered,263
Male,Stable,197


**Discussão:** Os resultados demonstram que o desfecho de recuperação plena ("Recovered") é o cenário predominante para ambos os géneros. No entanto, regista-se uma diferença no volume e na proporção de sucesso: as pacientes do género feminino apresentam um número absoluto maior de recuperações (328 casos) face aos pacientes do género masculino (263 casos). Em termos práticos para a gestão hospitalar e para as equipas clínicas, isto sugere que os pacientes do género masculino apresentam uma taxa ligeiramente maior de estabilização ("Stable") sem recuperação total imediata. Consequentemente, o hospital pode utilizar esta informação para refinar os protocolos de acompanhamento, garantindo uma monitorização mais intensiva ou programas de reabilitação específicos para o público masculino, visando equiparar as taxas de recuperação plena.